## Importing libraries

In [17]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

## Getting data

In [18]:
def get_dfs():
    cwd = os.getcwd()
    path_to_data_dir = os.path.join(cwd, 'data', 'raw')

    if not os.path.exists(path_to_data_dir):
        raise FileNotFoundError(f"Directory not found: {path_to_data_dir}")

    data = [
        'customers.csv',
        'order_items.csv',
        'orders.csv',
        'products.csv',
    ]

    dfs_list = []
    for csv in data:
        file_path = os.path.join(path_to_data_dir, csv)
        df = pd.read_csv(file_path)
        dfs_list.append(df)

    return dfs_list

## Cleaning customers

In [19]:
def transform_customers(df):
    df_ = df.copy()
    
    df_ = (
        df_
        .drop_duplicates(subset=['customer_id'])
        .loc[lambda x: x['email'].str.contains('@', na=False)] 
        .assign(created_at=lambda df__: pd.to_datetime(df__['created_at'], errors='coerce'))
        .dropna()  
        .astype({'customer_id': 'int'})
        .set_index('customer_id')
    )
    
    return df_

Removes customer_id duplicates, keeps valid emails with @, converts created_at to datetime, drops NaN, sets customer_id as int index.

## Cleaning products

In [20]:
def transform_products(df):
    df_ = df.copy()
    
    df_ = (
        df_
        .drop_duplicates(subset=['product_id', 'name'])
        .dropna()
        .loc[lambda x: x['price'] > 0]
        .astype({'product_id': 'int', 'price': 'float'})
    )
    
    return df_

Removes product_id and (name duplicates in future), drops NaN, keeps price > 0, converts product_id to int and price to float.

## Cleaning order items

In [21]:
def transform_order_items(df):
    df_ = df.copy()
    
    df_ = (
        df_
        .drop_duplicates(subset=['order_item_id'])
        .dropna()
        .loc[lambda x: x['quantity'] > 0]
        .astype({'order_item_id': 'int', 'quantity': 'int'})
        .set_index('order_item_id')
    )
    
    return df_

Removes order_item_id duplicates, drops NaN, keeps quantity > 0, converts to int types, sets order_item_id as index.

## Cleaning orders

In [22]:
def transform_orders(df):
    df_ = df.copy()

    valid_statuses = ['pending', 'completed', 'returned', 'cancelled']

    df_ = (
        df_
        .drop_duplicates(subset=['order_id'])
        .dropna()
        .assign(
            order_status=lambda x: x['order_status'].str.lower(),
            created_at=lambda x: pd.to_datetime(x['created_at'], errors='coerce')
        )
        .loc[lambda x: x['order_status'].isin(valid_statuses)]
        .astype({
            'order_id': 'int',
            'customer_id': 'int'
        })
        .set_index('order_id')
    )

    return df_

Removes order_id duplicates, normalizes order_status (strip+lower), converts created_at to datetime, filters valid statuses, converts order_id/customer_id to int index.

## Load df`s in DB

In [23]:
def load_to_postgres(df, table_name, engine, index=False, index_label=None):
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=index,
        index_label=index_label,
        method='multi'
    )

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "elt_pipeline")
DB_USER = os.getenv("DB_USER", "deructu")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

customers_df, order_items_df, orders_df, products_df = get_dfs()
customers_clean = transform_customers(customers_df)
products_clean = transform_products(products_df)
orders_clean = transform_orders(orders_df)
order_items_clean = transform_order_items(order_items_df)

load_to_postgres(customers_clean, 'clean_customers', engine, index=True, index_label='customer_id')
load_to_postgres(products_clean, 'clean_products', engine, index=False)
load_to_postgres(orders_clean, 'clean_orders', engine, index=True, index_label='order_id')
load_to_postgres(order_items_clean, 'clean_order_items', engine, index=True, index_label='order_item_id')

## Analytic tables

In [24]:
sql_queries = [
    """
    DROP TABLE IF EXISTS customer_summary;
    CREATE TABLE customer_summary AS
    SELECT 
        c.customer_id, c.email, c.country,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_spent
    FROM clean_customers c
    LEFT JOIN clean_orders o ON c.customer_id = o.customer_id
    LEFT JOIN clean_order_items oi ON o.order_id = oi.order_id
    LEFT JOIN clean_products p ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.email, c.country;
    """,
    """
    DROP TABLE IF EXISTS product_performance;
    CREATE TABLE product_performance AS
    SELECT 
        p.product_id, p.name, p.category,
        COALESCE(SUM(oi.quantity), 0) AS total_quantity_sold,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_revenue
    FROM clean_products p
    LEFT JOIN clean_order_items oi ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.name, p.category;
    """
]

with engine.connect() as conn:
    for q in sql_queries:
        conn.execute(text(q))
    conn.commit()

perf_df = pd.read_sql("SELECT * FROM product_performance ORDER BY total_revenue", engine)
cust_df = pd.read_sql("SELECT * FROM customer_summary ORDER BY total_spent", engine)

perf_df.sort_values(by='product_id').set_index('product_id')

,name,category,total_quantity_sold,total_revenue
product_id,,,,
1001,Toys Product 1001,Toys,12.0,9720.36
1002,Stationery Product 1002,Stationery,35.0,27902.70
1003,Stationery Product 1003,Stationery,17.0,21182.85
1004,Stationery Product 1004,Stationery,17.0,23311.25
1005,Duplicate Product 1005,Electronics,19.0,18999.81
...,...,...,...,...
1146,Stationery Product 1146,Stationery,2.0,1292.02
1147,Stationery Product 1147,Stationery,30.0,30539.10
1148,Sports Product 1148,Sports,27.0,19736.46


In [25]:
cust_df.sort_values(by='customer_id').set_index('customer_id')

,email,country,total_orders,total_spent
customer_id,,,,
1,user1@example.com,DE,1,2734.20
2,user2@example.com,PL,2,2713.70
3,user3@example.com,DE,1,0.00
4,user4@example.com,CZ,1,1206.74
5,user5@example.com,US,1,2411.51
...,...,...,...,...
346,user346@example.com,UA,1,3019.65
347,user347@example.com,US,3,10400.46
348,user348@example.com,SE,2,1027.90
